In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import random, math, time
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict


# =========================
# Knight Block (6x6) - Empty start
# Phase 1: Human places knight by click, then AI places knight randomly.
# Phase 2: Move like knight. Leaving cell becomes blocked forever.
# No capture. If you have no move on your turn -> you lose.
# Draw: 3-fold repetition OR move limit.
# AI: Adversarial MCTS (UCT) + smart rollouts.
# =========================

SIZE = 6
KDIRS = [(2, 1), (2, -1), (-2, 1), (-2, -1),
         (1, 2), (1, -2), (-1, 2), (-1, -2)]

HUMAN = 1
AI = 2

# Draw controls
MAX_PLIES = 140
REPEAT_LIMIT = 3


def inb(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE


def rc(pos: int) -> Tuple[int, int]:
    return divmod(pos, SIZE)


def pidx(r, c) -> int:
    return r * SIZE + c


def knight_moves(pos: int) -> List[int]:
    r, c = rc(pos)
    out = []
    for dr, dc in KDIRS:
        nr, nc = r + dr, c + dc
        if inb(nr, nc):
            out.append(pidx(nr, nc))
    return out


@dataclass(frozen=True)
class StateKey:
    h: int
    a: int
    blocked: int
    turn: int


@dataclass
class State:
    h: int         # human position (-1 if not placed yet)
    a: int         # ai position (-1 if not placed yet)
    blocked: int   # bitmask 36 bits
    turn: int      # HUMAN or AI
    ply: int       # half-moves so far


def bit_has(mask: int, pos: int) -> bool:
    return (mask >> pos) & 1 == 1


def bit_set(mask: int, pos: int) -> int:
    return mask | (1 << pos)


def legal_moves_state(s: State) -> List[int]:
    me = s.h if s.turn == HUMAN else s.a
    opp = s.a if s.turn == HUMAN else s.h
    if me < 0 or opp < 0:
        return []

    moves = []
    for nxt in knight_moves(me):
        if nxt == opp:
            continue
        if bit_has(s.blocked, nxt):
            continue
        moves.append(nxt)
    return moves


def apply_move_state(s: State, dest: int) -> State:
    # leaving cell becomes blocked
    if s.turn == HUMAN:
        leave = s.h
        new_blocked = bit_set(s.blocked, leave)
        return State(h=dest, a=s.a, blocked=new_blocked, turn=AI, ply=s.ply + 1)
    else:
        leave = s.a
        new_blocked = bit_set(s.blocked, leave)
        return State(h=s.h, a=dest, blocked=new_blocked, turn=HUMAN, ply=s.ply + 1)


def terminal_result(s: State) -> Optional[int]:
    # draw by move limit
    if s.ply >= MAX_PLIES:
        return 0

    lm = legal_moves_state(s)
    if not lm:
        # current player to move loses
        return +1 if s.turn == HUMAN else -1
    return None


# =========================
# Rollout / Heuristic
# =========================
def mobility_score(s: State) -> int:
    t = s.turn

    s.turn = AI
    ai_m = len(legal_moves_state(s))
    s.turn = HUMAN
    hu_m = len(legal_moves_state(s))

    s.turn = t
    return ai_m - hu_m


def rollout_policy_move(s: State) -> int:
    moves = legal_moves_state(s)
    if not moves:
        return -1

    best = []
    best_val = -10**9

    for m in moves:
        ns = apply_move_state(s, m)

        # If opponent has no moves next => immediate win
        tr = terminal_result(ns)
        if tr is not None:
            return m

        val = mobility_score(ns)

        # avoid likely self-trap
        opp_moves = legal_moves_state(ns)
        if opp_moves:
            ns2 = apply_move_state(ns, random.choice(opp_moves))
            my_next = len(legal_moves_state(ns2))
            if my_next == 0:
                val -= 60

        if val > best_val:
            best_val = val
            best = [m]
        elif val == best_val:
            best.append(m)

    return random.choice(best) if best else random.choice(moves)


def rollout(s: State, rep_counter: Dict[StateKey, int]) -> float:
    cur = State(s.h, s.a, s.blocked, s.turn, s.ply)
    local_rep = dict(rep_counter)

    while True:
        key = StateKey(cur.h, cur.a, cur.blocked, cur.turn)
        local_rep[key] = local_rep.get(key, 0) + 1
        if local_rep[key] >= REPEAT_LIMIT:
            return 0.0

        tr = terminal_result(cur)
        if tr is not None:
            return float(tr)

        mv = rollout_policy_move(cur)
        if mv == -1:
            return 0.0
        cur = apply_move_state(cur, mv)


# =========================
# Adversarial MCTS (UCT)
# =========================
class Node:
    __slots__ = ("state", "parent", "move", "children", "untried", "visits", "value")

    def __init__(self, state: State, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children: List["Node"] = []
        self.untried = legal_moves_state(state)
        self.visits = 0
        self.value = 0.0  # AI perspective

    def uct(self, child, c=1.35):
        if child.visits == 0:
            return float("inf")
        exploit = child.value / child.visits
        explore = c * math.sqrt(math.log(self.visits + 1) / child.visits)
        return exploit + explore

    def select_child_adversarial(self, c=1.35):
        # AI nodes -> maximize, Human nodes -> minimize
        if self.state.turn == AI:
            return max(self.children, key=lambda ch: self.uct(ch, c))
        else:
            return min(self.children, key=lambda ch: self.uct(ch, c))

    def add_child(self, move, next_state):
        child = Node(next_state, parent=self, move=move)
        for i, m in enumerate(self.untried):
            if m == move:
                self.untried.pop(i)
                break
        self.children.append(child)
        return child

    def update(self, reward):
        self.visits += 1
        self.value += reward


def mcts_best_move(root_state: State, root_rep: Dict[StateKey, int], time_limit=2.0) -> int:
    root = Node(root_state)
    start = time.time()

    while (time.time() - start) < time_limit:
        node = root
        state = State(root_state.h, root_state.a, root_state.blocked, root_state.turn, root_state.ply)
        rep = dict(root_rep)

        # Selection
        while not node.untried and node.children:
            node = node.select_child_adversarial()
            state = apply_move_state(state, node.move)
            k = StateKey(state.h, state.a, state.blocked, state.turn)
            rep[k] = rep.get(k, 0) + 1
            if rep[k] >= REPEAT_LIMIT:
                break

        # Terminal / repetition
        tr = terminal_result(state)
        if tr is None:
            k = StateKey(state.h, state.a, state.blocked, state.turn)
            if rep.get(k, 0) >= REPEAT_LIMIT:
                tr = 0

        # Expansion
        if tr is None and node.untried:
            # bias to better mobility
            scored = []
            for m in node.untried:
                ns = apply_move_state(state, m)
                scored.append((mobility_score(ns), m))
            scored.sort(reverse=True)
            top = [m for _, m in scored[:min(6, len(scored))]]
            m = random.choice(top)

            state = apply_move_state(state, m)
            k2 = StateKey(state.h, state.a, state.blocked, state.turn)
            rep[k2] = rep.get(k2, 0) + 1
            node = node.add_child(m, state)

        # Simulation
        if tr is None:
            reward = rollout(state, rep)
        else:
            reward = float(tr)

        # Backprop
        while node is not None:
            node.update(reward)
            node = node.parent

    if not root.children:
        ms = legal_moves_state(root_state)
        return random.choice(ms) if ms else -1
    best = max(root.children, key=lambda ch: ch.visits)
    return best.move


# =========================
# GUI with placement phase
# =========================
class KnightBlockGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Knight Block 6x6 — Empty Start (Place then Play)")

        top = tk.Frame(root)
        top.pack(fill="x", padx=10, pady=6)

        tk.Label(top, text="AI Think Time (sec):").pack(side="left")
        self.think_time = tk.DoubleVar(value=2.0)
        ttk.Scale(top, from_=0.6, to=4.0, orient="horizontal", length=200, variable=self.think_time).pack(side="left", padx=8)

        tk.Button(top, text="New Game", command=self.new_game).pack(side="right")

        self.status = tk.StringVar(value="")
        tk.Label(root, textvariable=self.status, fg="blue").pack(pady=(0, 6))

        self.cell = 80
        self.pad = 20
        w = self.pad * 2 + self.cell * SIZE
        h = self.pad * 2 + self.cell * SIZE
        self.canvas = tk.Canvas(root, width=w, height=h, bg="white")
        self.canvas.pack(padx=10, pady=10)

        self.canvas.bind("<Button-1>", self.on_click)

        self.new_game()

    def new_game(self):
        # Start completely empty:
        self.state = State(h=-1, a=-1, blocked=0, turn=HUMAN, ply=0)
        self.phase = "PLACE_HUMAN"  # PLACE_HUMAN -> PLACE_AI -> PLAY
        self.ai_busy = False

        self.rep: Dict[StateKey, int] = {}  # repetition only starts when both placed

        self.draw()
        self.status.set("Click ANY cell to place your knight (Human).")

    def start_play_phase(self):
        self.phase = "PLAY"
        # initialize repetition map now that positions are valid
        self._touch_rep()
        self.draw()
        self.status.set("Your turn. Click a green square to move.")

        # if human has no move (rare), human loses immediately
        tr = terminal_result(self.state)
        if tr is not None:
            self.finish_game(tr)

    def _touch_rep(self):
        k = StateKey(self.state.h, self.state.a, self.state.blocked, self.state.turn)
        self.rep[k] = self.rep.get(k, 0) + 1

    def finish_game(self, res):
        if res == 0:
            messagebox.showinfo("Game Over", "Draw!")
        elif res == 1:
            messagebox.showinfo("Game Over", "AI wins! 💀")
        else:
            messagebox.showinfo("Game Over", "You win! 🎉")

    def on_click(self, event):
        x, y = event.x, event.y
        c = (x - self.pad) // self.cell
        r = (y - self.pad) // self.cell
        if not inb(r, c):
            return
        pos = pidx(r, c)

        if self.phase == "PLACE_HUMAN":
            # place human
            self.state.h = pos
            self.phase = "PLACE_AI"

            # AI places randomly on an empty different cell
            choices = [p for p in range(SIZE * SIZE) if p != self.state.h]
            self.state.a = random.choice(choices)

            self.draw()
            self.status.set("AI placed its knight. Now the game starts: Your turn.")
            self.start_play_phase()
            return

        if self.phase != "PLAY":
            return

        if self.ai_busy or self.state.turn != HUMAN:
            return

        legal = legal_moves_state(self.state)
        if pos not in legal:
            return

        # Human move
        self.state = apply_move_state(self.state, pos)
        self._touch_rep()

        k = StateKey(self.state.h, self.state.a, self.state.blocked, self.state.turn)
        if self.rep.get(k, 0) >= REPEAT_LIMIT:
            self.draw()
            self.finish_game(0)
            return

        tr = terminal_result(self.state)
        self.draw()
        if tr is not None:
            self.finish_game(tr)
            return

        # AI move
        self.status.set("AI thinking...")
        self.ai_busy = True
        self.root.after(60, self.ai_turn)

    def ai_turn(self):
        if self.state.turn != AI:
            self.ai_busy = False
            return

        tr = terminal_result(self.state)
        if tr is not None:
            self.ai_busy = False
            self.finish_game(tr)
            return

        t = float(self.think_time.get())
        mv = mcts_best_move(self.state, self.rep, time_limit=t)

        if mv == -1:
            self.ai_busy = False
            self.finish_game(-1)
            return

        self.state = apply_move_state(self.state, mv)
        self._touch_rep()

        k = StateKey(self.state.h, self.state.a, self.state.blocked, self.state.turn)
        self.draw()

        if self.rep.get(k, 0) >= REPEAT_LIMIT:
            self.ai_busy = False
            self.finish_game(0)
            return

        tr = terminal_result(self.state)
        if tr is not None:
            self.ai_busy = False
            self.finish_game(tr)
            return

        self.ai_busy = False
        self.status.set("Your turn. Click a green square to move.")

    def draw(self):
        self.canvas.delete("all")

        # board + blocked cells
        for r in range(SIZE):
            for c in range(SIZE):
                pos = pidx(r, c)
                x1 = self.pad + c * self.cell
                y1 = self.pad + r * self.cell
                x2 = x1 + self.cell
                y2 = y1 + self.cell

                base = "#f7f7f7" if (r + c) % 2 == 0 else "#ffffff"
                if self.state.blocked != 0 and bit_has(self.state.blocked, pos):
                    base = "#333333"
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=base, outline="#cccccc")

        # highlight legal moves for human during play
        if self.phase == "PLAY" and self.state.turn == HUMAN:
            for dest in legal_moves_state(self.state):
                rr, cc = rc(dest)
                x1 = self.pad + cc * self.cell
                y1 = self.pad + rr * self.cell
                x2 = x1 + self.cell
                y2 = y1 + self.cell
                self.canvas.create_rectangle(x1, y1, x2, y2, outline="#00aa00", width=3)

        # draw pieces if placed
        if self.state.h >= 0:
            self.draw_piece(self.state.h, "blue", "H♞")
        if self.state.a >= 0:
            self.draw_piece(self.state.a, "red", "A♞")

    def draw_piece(self, pos, color, label):
        r, c = rc(pos)
        cx = self.pad + c * self.cell + self.cell / 2
        cy = self.pad + r * self.cell + self.cell / 2
        rad = self.cell * 0.32
        self.canvas.create_oval(cx - rad, cy - rad, cx + rad, cy + rad, fill=color, outline="")
        self.canvas.create_text(cx, cy, text=label, fill="white", font=("Arial", 20, "bold"))


if __name__ == "__main__":
    root = tk.Tk()
    app = KnightBlockGUI(root)
    root.mainloop()
